# 3장 — 고무줄 자: 부동소수점 (핵심) 📏

웹 챕터에서 해부한 fp32 를 이번엔 **비트 수준에서 직접** 열어 봅니다.
부호·지수·가수를 추출해 복원 공식으로 되살리고, 눈금 간격까지 실측합니다.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: 워밍업의 세 미스터리(0.1 티끌·1 삼키기·늘어나는 눈금)를
**fp32 = 부호 1 + 지수 8 + 가수 23** 구조로 전부 풀어낸다.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch3')

## 1. fp32 해부: 0.1을 비트 수준에서 열어 본다

`view(np.uint32)` 로 (2장의 그 재주: 비트는 그대로, 약속만 바꿔) fp32 의 32개 스위치를
정수처럼 읽은 뒤, 세 부품으로 자릅니다.

In [ ]:
import numpy as np
from decimal import Decimal

f = np.array(0.1, dtype=np.float32)          # 0.1 을 fp32 칸에 담는다(반올림 발생)
u = int(f.view(np.uint32))                    # 같은 비트를 정수로 재독 — 2장의 view 재주
bits = f'{u:032b}'
s, e, m = u >> 31, (u >> 23) & 0xFF, u & 0x7FFFFF

print(f'32비트 전체: {bits[0]} | {bits[1:9]} | {bits[9:]}')
print(f'부호 s = {s} (+)   지수 e = {int(bits[1:9], 2)} − 127 = {e - 127}   가수 m = {m:,}')

# 복원 공식 — 세 부품으로 저장값을 되살린다: (−1)^s × (1 + m/2²³) × 2^(e−127)
rebuilt = (-1) ** s * (1 + m / 2**23) * 2.0 ** (e - 127)
print(f'복원값     = {rebuilt!r}')
print(f'실제 저장값 = {Decimal(float(f))}')

assert e - 127 == -4, '0.1 ≈ 1.6 × 2⁻⁴ — 지수부는 123(= −4 + bias 127)'
assert rebuilt == float(f), '세 부품 → 복원 공식이 저장값과 비트까지 일치'
assert Decimal(float(f)) != Decimal('0.1'), '저장값은 0.100000001490… — 워밍업 티끌의 진범은 가수 반올림'
print()
print('✔ 확인: fp32 = 부호 1 + 지수 8 + 가수 23. 공식 하나로 완전히 복원된다')

## 2. 눈금 간격 실측: 고무줄과 정수 천장

`np.spacing(x)` = x 에서 이웃 눈금까지의 거리. fp32 와 fp64(워밍업의 자)를 같은 자리에서 재 봅니다.

In [ ]:
print(' 수의 크기     fp32 간격        fp64 간격')
for x in [1.0, 1000.0, 1e7, 1e8]:
    print(f' {x:>10,.0f}   {np.spacing(np.float32(x)):12.6g}   {np.spacing(np.float64(x)):12.6g}')

assert np.spacing(np.float32(1.0)) == 2.0 ** -23, '1 근처 fp32 간격 = 2⁻²³ — 유효숫자 약 7자리의 근원'
assert np.spacing(np.float32(2**23)) == 1.0 and np.spacing(np.float32(2**24)) == 2.0, '창문이 2배면 간격도 2배'
assert np.float32(2**24) + np.float32(1) == np.float32(2**24), 'fp32 의 1 삼키기 — 경계 2²⁴ = 16,777,216'
assert np.float64(2**24) + 1 == 2**24 + 1, '같은 자리에서 fp64 는 여유(경계는 2⁵³ ≈ 9천조 — 워밍업)'
assert float(np.float32(12345678.9)) == 12345679.0, '간격 1 구간 — 소수점 증발'
print()
print('✔ 확인: 두 자 모두 고무줄(간격 ∝ 크기). 정수 천장은 가수 비트 수가 정한다 (fp32: 2²⁴, fp64: 2⁵³)')

## 3. 두 고무줄을 겹쳐 그리기

로그-로그에서 두 직선이 **같은 기울기, 다른 높이**. 구조는 같고 눈금 밀도만 다르다는 증거입니다.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

xs = 10.0 ** np.arange(-2, 10)
g32 = np.spacing(xs.astype(np.float32)).astype(float)
g64 = np.spacing(xs)

plt.figure(figsize=(6.5, 3.8))
plt.loglog(xs, g32, 'o-', color='tab:purple', label='fp32 (가수 23비트)')
plt.loglog(xs, g64, 's-', color='tab:gray', label='fp64 (가수 52비트, 워밍업의 자)')
plt.axhline(1, color='tab:red', ls='--', label='간격 = 1 (+1 삼키기 경계)')
plt.xlabel('수의 크기'); plt.ylabel('이웃 눈금까지의 간격'); plt.legend(fontsize=8); plt.grid(alpha=0.3, which='both')
plt.title('같은 고무줄, 다른 밀도: fp32 vs fp64'); plt.tight_layout(); plt.show()

assert np.all(g32 > g64), '같은 자리에서 fp32 간격이 항상 더 넓다 — 가수가 29비트 적으니까'
print('✔ 확인: 기울기(고무줄 구조)는 같고 높이(밀도)만 다르다. 칸 크기를 정하는 건 가수·지수 비트 배분')

**직접 해보기**: ① 1절의 `0.1` 을 `0.5` 로 바꾸면 가수가 전부 0이 되고 오차도 사라지는 걸
확인해 보세요(눈금 위의 수). ② 7장 예고편: `Decimal(float(np.float16(0.1)))` 을 실행해 보세요.
칸을 반(fp16)으로 줄이면 0.1 의 티끌이 0.0999755859375 로 **6만 배쯤 커집니다.**

## 4. 관찰 정리 + 🤖 AI 연결

- **fp32 = 부호 1 + 지수 8 + 가수 23**: (−1)^s × 1.m × 2^(e−127). 숨은 1 덕에 실질 24비트.
- **지수 = 자의 길이**(범위 ±3.4×10³⁸, bias 127 약속), **가수 = 눈금 촘촘함**(유효숫자 약 7자리).
- 지수가 고른 창문을 가수가 2²³ 칸으로 나눔 → 창문 2배마다 간격 2배(고무줄) →
  **상대 정밀도(≈2⁻²⁴)는 어디서나 일정.** 정수 천장: fp32 = 2²⁴(1,677만), fp64 = 2⁵³(9천조).
- 워밍업 수수께끼 풀이 완료: 0.1 티끌 = 가수 반올림, 1 삼키기 = 정수 천장, 늘어나는 눈금 = 창문 구조.
- **다시 쓰는 수학**: 2부 7장(지수: 범위의 언어), 5부 1장(log₂: 비트 수 감각).
- **7장 예고**: fp16(지수5+가수10)·bf16(지수8+가수7)은 같은 세 부품의 **비트 재배분**.
  자의 길이를 지킬까, 눈금을 지킬까의 선택.

**다음 챕터**: 4장(눈금 사이의 오차). 티끌이 수십억 번 쌓이면? 더하는 순서만 바꿔도
답이 달라지는 세계(GPU 재현성의 비밀)로 갑니다.